# Task 2: Predict Future Stock Prices (Corrected for Future Prediction)

**Objective**: Use historical stock data to predict the **next day's** closing price.

### Step 1: Import Dependencies

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

### Step 2: Fetch Data and Prepare Lagged Target
To predict the *future*, we shift the 'Close' price by -1. This means for each day, the target (y) is the closing price of the **following day**.

In [ ]:
stock_symbol = "TSLA"
data = yf.download(stock_symbol, start="2019-01-01", end="2024-01-01")

# Create the 'Prediction' column (Tomorrow's Close)
data['Next_Close'] = data['Close'].shift(-1)

# Drop the last row because we don't know what happened after the end date
data.dropna(inplace=True)

print("Sample data with Next_Close target:")
print(data[['Close', 'Next_Close']].head())

### Step 3: Train/Test Split (Time-Series Safe)
We use `shuffle=False` to ensure no future data leaks into the training phase.

In [ ]:
features = ['Open', 'High', 'Low', 'Close', 'Volume']
X = data[features]
y = data['Next_Close']

# Split: 80% past data for training, 20% recent data for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

print(f"Training range: {X_train.index[0]} to {X_train.index[-1]}")
print(f"Testing range: {X_test.index[0]} to {X_test.index[-1]}")

### Step 4: Model Training and Evaluation
Linear Regression is very fast because it uses a closed-form mathematical solution (Ordinary Least Squares). This speed is normal for small datasets.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")

# Plotting
plt.figure(figsize=(14, 7))
plt.plot(y_test.index, y_test.values, label='Actual Next Day Close', color='blue')
plt.plot(y_test.index, y_pred, label='Predicted Next Day Close', color='red', alpha=0.8)
plt.title(f"{stock_symbol} Future Price Prediction")
plt.legend()
plt.show()